In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib.gridspec as gridspec

from scipy.stats import norm as sp_norm
from scipy.optimize import differential_evolution
from scipy.special import rel_entr

## Read Data
use two year data for post-covid representation

In [21]:
# Load Data
data2023        = pd.read_csv('atusact-2023/atusact_2023.dat',   dtype={'TRCODE': str, 'TUCASEID': str})
respondents2023 = pd.read_csv('atusresp-2023/atusresp_2023.dat', dtype={'TUCASEID': str})
jointed2023     = pd.merge(data2023, respondents2023, on='TUCASEID', how='inner')

data2024        = pd.read_csv('atusact-2024/atusact_2024.dat',   dtype={'TRCODE': str, 'TUCASEID': str})
respondents2024 = pd.read_csv('atusresp-2024/atusresp_2024.dat', dtype={'TUCASEID': str})
jointed2024     = pd.merge(data2024, respondents2024, on='TUCASEID', how='inner')

joined_data = pd.concat([jointed2023, jointed2024], ignore_index=True)

keep_cols = [
    'TUCASEID', 'TUDIARYDAY', 'TUFINLWGT',
    'TESCHENR', 'TESCHFT', 'TESCHLVL',
    'TRHHCHILD', 'TRDPFTPT', 'TELFS', 'TRTHH',
    'TRCODE', 'TUACTDUR', 'TUSTARTTIM', 'TUSTOPTIME', 'TEWHERE',
]
joined_data = joined_data[[c for c in keep_cols if c in joined_data.columns]]

# Day type
joined_data['daytype'] = joined_data['TUDIARYDAY'].map(
    lambda d: 'weekend' if d in (1, 7) else 'weekday'
)
print(f'Loaded {joined_data["TUCASEID"].nunique():,} respondents, {len(joined_data):,} activity rows')

Loaded 16,217 respondents, 292,655 activity rows


## Clean Data for 3 state model

In [22]:
def map_state(row):
    if str(row['TRCODE']).startswith('0101'):
        return 'sleep'
    elif row['TEWHERE'] == 1:
        return 'home'
    else:
        return 'away'

joined_data['state'] = joined_data.apply(map_state, axis=1)

joined_data['daytype'] = joined_data['TUDIARYDAY'].map(
    lambda d: 'weekend' if d in (1, 7) else 'weekday'
)

In [26]:
DIARY_START = 4 * 60  # 4am = 240 min; this is t=0 in our coordinate system

def parse_atus_time(t):
    """Parse 'HH:MM' or 'HH:MM:SS' → minutes since 4am. Times before 4am are post-midnight (+1440)."""
    parts = str(t).strip().split(':')
    h, m = int(parts[0]), int(parts[1])
    minutes = h * 60 + m
    if minutes < DIARY_START:
        minutes += 1440          # post-midnight, belongs to next side of diary
    return minutes - DIARY_START  # shift so 4am = 0

joined_data['start_min'] = joined_data['TUSTARTTIM'].apply(parse_atus_time)
joined_data['stop_min']  = joined_data['TUSTOPTIME'].apply(parse_atus_time)

# If an activity crosses midnight and ends after 4am, stop_min will appear < start_min
# (e.g. sleep 9pm→8am: start=1032, stop=240). Add 1440 to fix.
mask = joined_data['stop_min'] < joined_data['start_min']
joined_data.loc[mask, 'stop_min'] += 1440

# Sort activities within each respondent
joined_data = joined_data.sort_values(['TUCASEID', 'start_min']).reset_index(drop=True)

# Identify consecutive same-state runs per respondent
joined_data['state_grp'] = (
    joined_data.groupby('TUCASEID')['state']
    .transform(lambda s: (s != s.shift()).cumsum())
)

# Collapse into state episodes
state_episodes = (
    joined_data
    .groupby(['TUCASEID', 'daytype', 'state_grp'], sort=False)
    .agg(
        state    = ('state', 'first'),
        start_min= ('start_min', 'min'),
        stop_min = ('stop_min',  'max'),
    )
    .reset_index()
    .drop(columns='state_grp')
)
state_episodes['duration'] = state_episodes['stop_min'] - state_episodes['start_min']


In [31]:
STATE_MAP  = {'sleep': 0, 'home': 1, 'away': 2}
DIARY_LEN  = 24 * 60  # 1440 bins, one per minute from 4am to 4am
BIN_SIZE   = 15       # minutes per coarse bin
N_BINS     = DIARY_LEN // BIN_SIZE  # 96 bins

def episodes_to_vector(episodes):
    """Convert a respondent's state episodes into a 1440-bin array (cap at 1440)."""
    vec = np.full(DIARY_LEN, -1, dtype=np.int8)  # -1 = unrecorded
    for _, row in episodes.iterrows():
        start = int(np.clip(row['start_min'], 0, DIARY_LEN))
        stop  = int(np.clip(row['stop_min'],  0, DIARY_LEN))
        vec[start:stop] = STATE_MAP[row['state']]
    return vec

def coarsen_vector(vec):
    """Downsample 1440-bin vector to 96 bins by majority vote (ignoring -1)."""
    coarse = np.full(N_BINS, -1, dtype=np.int8)
    for i in range(N_BINS):
        window = vec[i * BIN_SIZE : (i + 1) * BIN_SIZE]
        recorded = window[window >= 0]
        if len(recorded) > 0:
            coarse[i] = np.bincount(recorded, minlength=3).argmax()
    return coarse

records = []
for (caseid, daytype), grp in state_episodes.groupby(['TUCASEID', 'daytype']):
    vec    = episodes_to_vector(grp)
    coarse = coarsen_vector(vec)
    records.append({'TUCASEID': caseid, 'daytype': daytype, 'vec': coarse})

state_vectors = pd.DataFrame(records)
print(state_vectors.shape)
state_vectors.head()


(16217, 3)


,TUCASEID,daytype,vec
0,20230101230035,weekday,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,20230101230077,weekend,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,20230101230105,weekend,"[0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 2, 2, 1, 1, ..."
3,20230101230522,weekend,"[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,20230101230524,weekday,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


## Clustering — Weekday vs Weekend

In [32]:
from tslearn.clustering import TimeSeriesKMeans
from sklearn.metrics import silhouette_score

def prepare_matrix(df_daytype):
    """Stack coarse vectors, fill gaps via ffill/bfill, one-hot encode → (n, 96, 3)."""
    vecs = np.stack(df_daytype['vec'].values).astype(np.float32)  # (n, 96)

    # Fill -1 gaps: forward then backward along time axis
    for i in range(vecs.shape[0]):
        row = vecs[i]
        mask = row >= 0
        if mask.sum() == 0:
            continue  # all unrecorded — will be dropped below
        idx = np.arange(len(row))
        # forward fill
        last = row[mask][0]
        for j in range(len(row)):
            if row[j] >= 0:
                last = row[j]
            else:
                row[j] = last
        vecs[i] = row

    # Drop respondents still containing -1 (all-unrecorded)
    valid = ~np.any(vecs < 0, axis=1)
    vecs  = vecs[valid]
    caseids = df_daytype['TUCASEID'].values[valid]

    # One-hot encode → (n, 96, 3)
    n = vecs.shape[0]
    onehot = np.zeros((n, N_BINS, 3), dtype=np.float32)
    for s in range(3):
        onehot[:, :, s] = (vecs == s)

    return onehot, caseids

wd_df = state_vectors[state_vectors['daytype'] == 'weekday'].reset_index(drop=True)
we_df = state_vectors[state_vectors['daytype'] == 'weekend'].reset_index(drop=True)

X_wd, ids_wd = prepare_matrix(wd_df)
X_we, ids_we = prepare_matrix(we_df)

print(f'Weekday matrix: {X_wd.shape}')
print(f'Weekend matrix: {X_we.shape}')


/opt/anaconda3/envs/OB/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


Weekday matrix: (8203, 96, 3)
Weekend matrix: (8014, 96, 3)


In [ ]:
import pickle, os, time

K_RANGE      = range(2, 9)
ELBOW_SAMPLE = 4000
SAVE_DIR     = 'elbow_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
rng          = np.random.default_rng(42)

def elbow_silhouette(X, label):
    save_path = os.path.join(SAVE_DIR, f'{label}_elbow.pkl')

    # Resume from checkpoint if it exists
    if os.path.exists(save_path):
        with open(save_path, 'rb') as f:
            results = pickle.load(f)
        print(f'  Resumed {label} from checkpoint: {list(results.keys())} done')
    else:
        results = {}

    idx = rng.choice(len(X), size=min(ELBOW_SAMPLE, len(X)), replace=False)
    Xs  = X[idx]

    for k in K_RANGE:
        if k in results:
            print(f'  {label} k={k}: skipped (cached)')
            continue
        t0 = time.time()
        km = TimeSeriesKMeans(n_clusters=k, metric='softdtw', max_iter=10,
                              random_state=42, n_jobs=-1)
        labels = km.fit_predict(Xs)
        Xs_flat = Xs.reshape(len(Xs), -1)
        sil = silhouette_score(Xs_flat, labels)
        elapsed = time.time() - t0
        results[k] = {'inertia': km.inertia_, 'silhouette': sil, 'model': km}
        with open(save_path, 'wb') as f:
            pickle.dump(results, f)
        print(f'  {label} k={k}: inertia={km.inertia_:.1f}, silhouette={sil:.3f}  [{elapsed:.0f}s] — saved')

    inertias    = [results[k]['inertia']    for k in K_RANGE]
    silhouettes = [results[k]['silhouette'] for k in K_RANGE]
    return inertias, silhouettes

print('Weekday:')
wd_inertias, wd_sil = elbow_silhouette(X_wd, 'weekday')
print('\nWeekend:')
we_inertias, we_sil = elbow_silhouette(X_we, 'weekend')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, inertias, silhouettes, title in zip(
    axes,
    [wd_inertias, we_inertias],
    [wd_sil, we_sil],
    ['Weekday', 'Weekend']
):
    ax2 = ax.twinx()
    ax.plot(list(K_RANGE), inertias,     'b-o', label='Inertia')
    ax2.plot(list(K_RANGE), silhouettes, 'r-s', label='Silhouette')
    ax.set_xlabel('k'); ax.set_ylabel('Inertia', color='b')
    ax2.set_ylabel('Silhouette', color='r')
    ax.set_title(f'{title} (subsample n={ELBOW_SAMPLE})')
    ax.legend(loc='upper right'); ax2.legend(loc='center right')
plt.tight_layout()
plt.show()


In [ ]:
# Fit final models with chosen k (adjust after inspecting elbow plot)
K_WEEKDAY  = 5
K_WEEKEND  = 4

km_wd = TimeSeriesKMeans(n_clusters=K_WEEKDAY, metric='softdtw', max_iter=20, random_state=42, n_jobs=-1)
km_we = TimeSeriesKMeans(n_clusters=K_WEEKEND, metric='softdtw', max_iter=20, random_state=42, n_jobs=-1)

labels_wd = km_wd.fit_predict(X_wd)
labels_we = km_we.fit_predict(X_we)

print('Weekday cluster sizes:', np.bincount(labels_wd))
print('Weekend cluster sizes:', np.bincount(labels_we))


In [ ]:
# Plot cluster centroids as stacked area charts
STATE_LABELS = ['Sleep', 'Home', 'Away']
STATE_COLORS = ['#5b8db8', '#f4a261', '#e76f51']
xticks       = np.arange(0, N_BINS + 1, 8)           # every 2 hours
xlabels      = [f'{(4 + t * BIN_SIZE // 60) % 24:02d}:00' for t in xticks]

def plot_centroids(km, title):
    k = km.n_clusters
    fig, axes = plt.subplots(1, k, figsize=(4 * k, 3), sharey=True)
    if k == 1:
        axes = [axes]
    for c, ax in enumerate(axes):
        centroid = km.cluster_centers_[c]   # shape (96, 3)
        # Normalise so rows sum to 1 (handles floating point drift)
        centroid = centroid / centroid.sum(axis=1, keepdims=True).clip(1e-9)
        bottom = np.zeros(N_BINS)
        for s in range(3):
            ax.fill_between(range(N_BINS), bottom, bottom + centroid[:, s],
                            color=STATE_COLORS[s], label=STATE_LABELS[s], alpha=0.85)
            bottom += centroid[:, s]
        ax.set_title(f'Cluster {c+1}')
        ax.set_xticks(xticks); ax.set_xticklabels(xlabels, rotation=45, ha='right')
        ax.set_ylim(0, 1); ax.set_xlim(0, N_BINS)
        if c == 0:
            ax.set_ylabel('State probability')
    handles = [plt.Rectangle((0,0),1,1, color=STATE_COLORS[s]) for s in range(3)]
    fig.legend(handles, STATE_LABELS, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))
    fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

plot_centroids(km_wd, 'Weekday Clusters')
plot_centroids(km_we, 'Weekend Clusters')


In [ ]:
o